# 01 · Los datos

Acá cargamos Home Credit, definimos quién es thin-file, partimos la muestra en train / calib / test y dejamos todo procesado (sin nulos, escalado y con las variables de texto pasadas a números), para que todos los modelos partan exactamente de los mismos datos.

- **Lee:** `data/application_train.csv` y `data/bureau.csv` (se bajan de Kaggle, ver `REPRODUCIBILIDAD.md`).
- **Guarda:** `outputs/datos_procesados.npz`, `outputs/ids_*.csv`, `outputs/preproc.joblib` y `outputs/nombres_columnas.csv`.
- **Números del informe que salen de acá:** el 14% de thin-file, la mora de 10,1% contra 7,7%, los 307.511 solicitantes, los 184.506 de train y las 200 variables.

In [1]:
# Bloque 1. Cargamos la base de solicitudes. Cada fila es un solicitante y TARGET = 1 quiere decir que cayó en mora.
import pandas as pd

df=pd.read_csv("data/application_train.csv")
df.shape

# Son 307.511 solicitantes y 122 columnas.

(307511, 122)

In [2]:
# miramos las primeras filas para tener una idea de qué hay en cada columna
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
# también (como son muchas variables), contamos cuántas son texto, pues tendremos que transformarlas más tarde
df.dtypes.value_counts()

# vemos que hay 16 de texto (str), el resto son números

float64    65
int64      41
str        16
Name: count, dtype: int64

In [4]:
# chequeamos la proporción de morosos y vemos los nulos

df["TARGET"].mean()

# la idea es ver el porcentaje de nulos por columna, ordenado de las que más a las que menos tienen (ascending=False),
# mostrando las primeras 15 (head)
(df.isna().mean()*100).sort_values(ascending=False).head(15)

COMMONAREA_AVG              69.872297
COMMONAREA_MODE             69.872297
COMMONAREA_MEDI             69.872297
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
NONLIVINGAPARTMENTS_AVG     69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_AVG               67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_AVG             66.497784
YEARS_BUILD_MODE            66.497784
dtype: float64

In [5]:
# las columnas con más faltantes son las que tienen que ver con la vivienda del solicitante y, a su vez, con resúmenes
# de la misma información (promedio, moda y mediana del edificio), por lo que inferimos que hay mucha redundancia

(df.isna().mean()>0.5).sum()   # hay 41 columnas a las que les falta más de la mitad de los datos

df["TARGET"].value_counts()   # importante el () para que me liste los conteos

# 282.686 pagaron y 24.825 cayeron en mora (8,07%).
# NOTA: los nulos NO se imputan con toda la base (eso sería data leakage, el modelo "vería" información de test).
# Primero partimos en train / calib / test y recién ahí aprendemos todo SOLO con train, para después aplicarlo
# igual a calib y test.

TARGET
0    282686
1     24825
Name: count, dtype: int64

In [6]:
# Bloque 2. ¿Quiénes son los thin-file? Cargamos el historial de bureau (lo anterior eran los datos de la solicitud;
# esto es el historial crediticio previo de cada cliente en otras entidades).

bureau=pd.read_csv("data/bureau.csv")

bureau.shape

(1716428, 17)

In [7]:
# hay 1.716.428 filas y 17 columnas: es historial previo, así que hay clientes que aparecen más de una vez
# (una fila por cada crédito anterior)

bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [8]:
# vemos cuántos clientes "únicos" hay en bureau
bureau["SK_ID_CURR"].nunique()

305811

In [9]:
# Ojo: bureau no cubre solo a los de application_train sino también a los de application_test (el archivo de Kaggle
# sin TARGET, que no usamos), por lo que no basta con restar conteos para tener a los thin-file.
# Por eso definimos thin_file con .isin(): es thin-file quien NO aparece en bureau, o sea quien pidió el crédito y
# no tiene ningún historial. El ~ invierte el True/False y astype(int) lo pasa a 1 (thin) y 0 (con historial).

df["thin_file"]=(~df["SK_ID_CURR"].isin(bureau["SK_ID_CURR"])).astype(int)

# ahora analizamos el segmento
df["thin_file"].value_counts()

# El aviso de PerformanceWarning es inofensivo (pandas sugiere armar las columnas de otra forma; no cambia nada).

/tmp/ipykernel_438/3779180422.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["thin_file"]=(~df["SK_ID_CURR"].isin(bureau["SK_ID_CURR"])).astype(int)


thin_file
0    263491
1     44020
Name: count, dtype: int64

In [10]:
# tasa de mora de cada segmento
df.groupby("thin_file")["TARGET"].mean()

thin_file
0    0.077301
1    0.101249
Name: TARGET, dtype: float64

In [11]:
# thin_file: 44.020 clientes (14,3% de la base) sin ningún registro en bureau.
# Tasa de mora: thin-file 10,1% contra 7,7% con historial (~31% más de riesgo relativo).
# Son minoría y más riesgosos. Ojo, es una diferencia descriptiva, no causal.
# Estos dos números (14% y 10,1% contra 7,7%) son los que aparecen en la introducción del informe.

# HIPÓTESIS (a testear, no conclusión): un modelo entrenado con toda la base podría ser sobreconfiado con los
# thin-file, porque son minoría y no tienen historial, y los modelos comunes dan probabilidades igual de "seguras"
# en zonas con pocos datos. Además, el promedio global puede verse bien y esconder el error del subgrupo, por eso
# todo se evalúa por segmento.

# Estrato: creamos una columna que identifica los 4 tipos de cliente (thin_file x TARGET),
# para que train, calib y test conserven las mismas proporciones de cada grupo:
# 0 = con historial, pagó
# 1 = con historial, moroso
# 2 = thin-file, pagó
# 3 = thin-file, moroso
# Sin estratificar, el grupo más chico (thin-file morosos, ~4.450) podría quedar desbalanceado entre las partes y
# romper la comparación por segmento.

df["estrato"] = 2 * df["thin_file"] + df["TARGET"]
df["estrato"].value_counts()

/tmp/ipykernel_438/3462416089.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["estrato"] = 2 * df["thin_file"] + df["TARGET"]


estrato
0    243123
2     39563
1     20368
3      4457
Name: count, dtype: int64

In [12]:
# Bloque 3. Partimos la muestra. Primero separamos el 30% de test y el 70% restante lo partimos después en train y
# calib. stratify=estrato mantiene la mezcla de los 4 grupos y random_state=42 hace que el corte sea siempre el mismo.
from sklearn.model_selection import train_test_split

no_test, test= train_test_split(df, test_size=0.3, stratify=df["estrato"], random_state=42)

In [13]:
print(len(no_test), len(test))

215257 92254


In [14]:
# nada se perdió ni se duplicó, pues suma los 307 mil

print(no_test["thin_file"].mean(), test["thin_file"].mean())

0.14314981626613768 0.14314826457389382


In [15]:
# aprox. el mismo promedio de thin_file en ambos, todo ok

# ahora, de lo que no va a test, queremos una parte en train y otra en calib (10/70 del 70% = 10% del total).
# En esta primera etapa, calib (10%) es el conjunto reservado para Conformal Selection. Más adelante (desde el 03)
# este 10% pasa a ser VALIDACIÓN y el 30% de test pasa a ser el POOL donde corre el experimento (ver README).

train, calib=train_test_split(no_test, test_size=10/70, stratify=no_test["estrato"],random_state=42)

In [16]:
print(len(train), len(calib), len(test))

184506 30751 92254


In [17]:
# 184.506 / 30.751 / 92.254, o sea 60% / 10% / 30%. Y la proporción de thin-file es la misma en las tres partes:
print(train["thin_file"].mean(), calib["thin_file"].mean(), test["thin_file"].mean())

0.14314981626613768 0.14314981626613768 0.14314826457389382


In [18]:
# Chequeo: ninguna fila puede estar en dos conjuntos a la vez (eso sería data leakage).
# index = número de fila que pandas le asigna a cada fila; al partir la tabla, cada fila conserva su índice original
# (por eso train.index tiene números salteados). set(...) arma un conjunto con los índices de cada parte, & devuelve
# los que tienen en común y len(...) cuenta cuántos hay. Los tres tienen que dar 0.
print(len(set(train.index) & set(test.index)))   # train y test
print(len(set(train.index) & set(calib.index)))   # train y calib
print(len(set(calib.index) & set(test.index)))   # calib y test

0
0
0


In [19]:
# ninguna observación (fila) está en dos muestras a la vez

# guardamos los ID que quedaron en cada muestra: todos los notebooks siguientes los usan para saber qué solicitante
# es cada fila, y así podemos cruzar las predicciones de los distintos modelos por ID y no por posición

train["SK_ID_CURR"].to_csv("outputs/ids_train.csv", index=False)   # index=False para que no guarde el número de fila como columna
calib["SK_ID_CURR"].to_csv("outputs/ids_calib.csv", index=False)
test["SK_ID_CURR"].to_csv("outputs/ids_test.csv", index=False)

pd.read_csv("outputs/ids_train.csv").shape

(184506, 1)

In [20]:
# Bloque 4. Pasamos a la etapa de preprocesamiento: primero buscamos anomalías mirando el resumen de cada variable

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", None)   # que pandas muestre todas las filas
train.describe().T

# DAYS_EMPLOYED tiene un máximo raro (365.243 días = 1.000 años de empleo). Lo miramos abajo.

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,"184,506.00","278,355.92","102,667.01","100,002.00","189,509.25","278,450.50","367,248.75","456,255.00"
TARGET,"184,506.00",0.08,0.27,0.00,0.00,0.00,0.00,1.00
CNT_CHILDREN,"184,506.00",0.42,0.72,0.00,0.00,0.00,1.00,19.00
AMT_INCOME_TOTAL,"184,506.00","168,846.82","291,265.20","25,650.00","112,500.00","144,000.00","202,500.00","117,000,000.00"
AMT_CREDIT,"184,506.00","599,330.77","402,870.18","45,000.00","270,000.00","513,531.00","808,650.00","4,050,000.00"
AMT_ANNUITY,"184,496.00","27,091.79","14,485.82","1,980.00","16,506.00","24,880.50","34,596.00","258,025.50"
AMT_GOODS_PRICE,"184,354.00","538,498.74","369,681.83","45,000.00","238,500.00","450,000.00","679,500.00","4,050,000.00"
REGION_POPULATION_RELATIVE,"184,506.00",0.02,0.01,0.00,0.01,0.02,0.03,0.07
DAYS_BIRTH,"184,506.00","-16,037.60","4,361.67","-25,201.00","-19,683.75","-15,754.00","-12,424.00","-7,489.00"
DAYS_EMPLOYED,"184,506.00","63,612.58","141,108.01","-17,912.00","-2,761.00","-1,216.00",-290.00,"365,243.00"


In [21]:
# cuántos tienen ese valor
(train["DAYS_EMPLOYED"]==365243).sum()

np.int64(33123)

In [22]:
# y qué proporción de train son
(train["DAYS_EMPLOYED"]==365243).mean()

np.float64(0.17952261715066176)

In [23]:
# ¿quiénes son? miramos su tipo de ingreso
train.loc[train["DAYS_EMPLOYED"] == 365243, "NAME_INCOME_TYPE"].value_counts()

# DAYS_EMPLOYED = 365243 es un código de Home Credit para "sin empleo actual" (no es una antigüedad real).
# Son 33.123 clientes en train (17,95%): 33.110 jubilados y 13 desempleados.
# Creamos una marca "sin_empleo" (para no perder la información) y después reemplazamos el 365243 por NaN.
# El orden importa: primero la marca, si no ya no se puede identificar a quién le pasaba.
# Es una regla fija de la documentación (no se calcula de los datos), así que se aplica igual a train, calib y test
# sin riesgo de leakage.

NAME_INCOME_TYPE
Pensioner     33110
Unemployed       13
Name: count, dtype: int64

In [24]:
import numpy as np

for d in [train, calib, test]:   # para cada muestra
    d["sin_empleo"]=(d["DAYS_EMPLOYED"]==365243).astype(int)   # si coincide, vale 1 en una columna nueva que llamamos "sin_empleo"
    d["DAYS_EMPLOYED"]= d["DAYS_EMPLOYED"].replace(365243, np.nan)   # después de usarlo, lo cambiamos por NaN (se imputa más adelante)

# ahora vemos cómo quedaron las medias y máximos
train["sin_empleo"].mean()

/tmp/ipykernel_438/1312399053.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["sin_empleo"]=(d["DAYS_EMPLOYED"]==365243).astype(int) # si coincide, que valga 1 en una columna q creamos y se llama "sin_empleo"
/tmp/ipykernel_438/1312399053.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["sin_empleo"]=(d["DAYS_EMPLOYED"]==365243).astype(int) # si coincide, que valga 1 en una columna q creamos y se llama "sin_empleo"
/tmp/ipykernel_438/1312399053.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually

np.float64(0.17952261715066176)

In [25]:
# el máximo ahora es 0 (los días de empleo se cuentan para atrás, en negativo): el código raro ya no está
train["DAYS_EMPLOYED"].max()

np.float64(0.0)

In [26]:
for d in [train, calib, test]:
    print(d["DAYS_EMPLOYED"].isna().sum(), d["sin_empleo"].sum())

# En las tres partes los NaN de DAYS_EMPLOYED coinciden exactamente con la marca sin_empleo.
# Por qué buscar anomalías: valores como DAYS_EMPLOYED = 365243 (un código, no un dato real) distorsionan medias,
# escalado y lo que aprende el modelo (leería "1.000 años de empleo"). Entenderlas antes de limpiar evita borrar
# información: acá el código significa "sin empleo".

33123 33123
5526 5526
16725 16725


In [27]:
# Bloque 5. Variables de texto: los modelos solo aceptan números, así que las 16 columnas de texto que identificamos
# antes hay que convertirlas. Primero vemos cuántas categorías tiene cada una (nunique) para elegir cómo codificarlas
# (pocas categorías: one-hot; muchas: agrupar las raras).

txt=train.select_dtypes(exclude="number").columns   # todo menos números
train[txt].nunique()

NAME_CONTRACT_TYPE             2
CODE_GENDER                    3
FLAG_OWN_CAR                   2
FLAG_OWN_REALTY                2
NAME_TYPE_SUITE                7
NAME_INCOME_TYPE               8
NAME_EDUCATION_TYPE            5
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
OCCUPATION_TYPE               18
WEEKDAY_APPR_PROCESS_START     7
ORGANIZATION_TYPE             58
FONDKAPREMONT_MODE             4
HOUSETYPE_MODE                 3
WALLSMATERIAL_MODE             7
EMERGENCYSTATE_MODE            2
dtype: int64

In [28]:
# Variables de texto (16 columnas), agrupadas por cantidad de categorías:
# - 2 categorías (NAME_CONTRACT_TYPE, FLAG_OWN_CAR, FLAG_OWN_REALTY, EMERGENCYSTATE_MODE): se pasan a 0/1.
# - 3 a 8 categorías (CODE_GENDER, NAME_INCOME_TYPE, NAME_FAMILY_STATUS, etc.): one-hot (una columna 0/1 por
#   categoría, así el modelo no asume un orden entre ellas).
# - Muchas categorías: OCCUPATION_TYPE (18) y ORGANIZATION_TYPE (58). Se agrupan las raras en una sola para no
#   crear decenas de columnas casi vacías.
# Anti-leakage: las categorías se aprenden solo con train (OneHotEncoder con fit en train) y se aplican igual a
# calib y test.
# Antes de decidir, revisamos las categorías raras de CODE_GENDER y los nulos de OCCUPATION_TYPE.

train["CODE_GENDER"].value_counts(dropna=False)

CODE_GENDER
F      121432
M       63072
XNA         2
Name: count, dtype: int64

In [29]:
train["OCCUPATION_TYPE"].value_counts(dropna=False)

OCCUPATION_TYPE
NaN                      57683
Laborers                 33224
Sales staff              19259
Core staff               16547
Managers                 12915
Drivers                  11154
High skill tech staff     6851
Accountants               5944
Medicine staff            5047
Security staff            4027
Cooking staff             3580
Cleaning staff            2774
Private service staff     1561
Low-skill Laborers        1254
Waiters/barmen staff       799
Secretaries                785
Realty agents              452
HR staff                   334
IT staff                   316
Name: count, dtype: int64

In [30]:
# ¿quiénes son los que no tienen ocupación cargada?

train.loc[train["OCCUPATION_TYPE"].isna(), "NAME_INCOME_TYPE"].value_counts()

# OCCUPATION_TYPE: 57.683 nulos (31,3%). El 57% de esos nulos son jubilados (sin ocupación actual, así que el nulo
# tiene significado) y ~25% son "Working" con el dato sin informar. Dejamos el nulo como su propia categoría
# ("missing") en vez de imputar, porque que falte ya es información. No agrupamos categorías acá: la más chica
# (IT staff) tiene 316 clientes, y de eso se encarga el min_frequency del one-hot.

NAME_INCOME_TYPE
Pensioner               33114
Working                 14949
Commercial associate     7347
State servant            2256
Unemployed                 13
Student                     2
Businessman                 2
Name: count, dtype: int64

In [31]:
train["ORGANIZATION_TYPE"].value_counts(dropna=False)

# ORGANIZATION_TYPE: 58 categorías con cola larga (~30 con menos de 1.000 clientes en train; la más chica tiene 15).
# Las raras se agrupan en una sola categoría con OneHotEncoder (min_frequency + handle_unknown="infrequent_if_exist"),
# con fit solo en train (sin leakage).
# XNA tiene 33.123 clientes, el mismo número que sin_empleo: otro código de "sin empleador". Lo verificamos abajo.

ORGANIZATION_TYPE
Business Entity Type 3    40722
XNA                       33123
Self-employed             22974
Other                     10062
Medicine                   6706
Business Entity Type 2     6338
Government                 6277
School                     5341
Trade: type 7              4743
Kindergarten               4104
Construction               4057
Business Entity Type 1     3618
Transport: type 4          3254
Trade: type 3              2104
Industry: type 9           2009
Industry: type 3           1935
Security                   1911
Housing                    1798
Industry: type 11          1646
Military                   1586
Bank                       1533
Agriculture                1494
Police                     1358
Postal                     1342
Transport: type 2          1338
Security Ministries        1191
Trade: type 2              1147
Restaurant                 1099
Services                    923
University                  797
Industry: type 7      

In [32]:
((train["ORGANIZATION_TYPE"] == "XNA") == (train["sin_empleo"] == 1)).mean()

# Da 1.0: ORGANIZATION_TYPE == "XNA" y sin_empleo == 1 son exactamente las mismas filas. Ambas marcan "sin empleador"
# (jubilados y desempleados). La columna one-hot de XNA va a duplicar a sin_empleo: es redundante pero inofensiva,
# así que la dejamos.

np.float64(1.0)

In [33]:
# Bloque 6. Listas de columnas por tipo, para usar después en el preprocesador:
# cat_cols = las 16 columnas de texto (se les aplica one-hot).
# num_cols = las numéricas, menos 3 que NO pueden ser variables predictoras: SK_ID_CURR (es solo un identificador),
#   TARGET (es lo que queremos predecir) y estrato (sale de TARGET: si entrara, el modelo vería la respuesta).
# thin_file y sin_empleo sí entran como variables.
# Chequeo: 16 + 106 = 122 columnas.

# categóricas
cat_cols=train.select_dtypes(exclude="number").columns.tolist()   # agarro los nombres de las que no son numéricas y armo una lista

excluir=["SK_ID_CURR", "TARGET", "estrato"]   # las que no pueden ser predictoras

# numéricas
num_cols=[]
for c in train.select_dtypes(include="number").columns:   # recorremos las columnas numéricas de train
    if c not in excluir:   # si no está en la lista de excluidas (o sea, es predictora)
        num_cols.append(c)   # la agregamos

In [34]:
print(len(cat_cols), len(num_cols), len(cat_cols) + len(num_cols))

# todo ok

16 106 122


In [35]:
# Bloque 7. Nulos en las numéricas: miramos cuántos hay y dónde antes de decidir cómo tratarlos.
# Por qué importa: XGBoost acepta nulos, pero la red neuronal y la capa GP de la SNGP no (un NaN rompe el cálculo),
# así que hay que completarlos. Y completar no es neutro: rellenar con la mediana dice "este cliente es típico" y
# puede borrar una señal si el dato faltaba por una razón (como la ocupación de los jubilados). Por eso decidimos
# según cuánto falta en cada columna.

nulos=train[num_cols].isna().mean().sort_values(ascending=False)   # proporción de nulos de cada numérica, de mayor a menor
nulos[nulos>0]   # solo las que tienen algún nulo

COMMONAREA_AVG                 0.70
COMMONAREA_MODE                0.70
COMMONAREA_MEDI                0.70
NONLIVINGAPARTMENTS_MODE       0.69
NONLIVINGAPARTMENTS_AVG        0.69
NONLIVINGAPARTMENTS_MEDI       0.69
LIVINGAPARTMENTS_MEDI          0.68
LIVINGAPARTMENTS_MODE          0.68
LIVINGAPARTMENTS_AVG           0.68
FLOORSMIN_AVG                  0.68
FLOORSMIN_MODE                 0.68
FLOORSMIN_MEDI                 0.68
YEARS_BUILD_MODE               0.67
YEARS_BUILD_AVG                0.67
YEARS_BUILD_MEDI               0.67
OWN_CAR_AGE                    0.66
LANDAREA_MEDI                  0.59
LANDAREA_MODE                  0.59
LANDAREA_AVG                   0.59
BASEMENTAREA_AVG               0.59
BASEMENTAREA_MEDI              0.59
BASEMENTAREA_MODE              0.59
EXT_SOURCE_1                   0.56
NONLIVINGAREA_MODE             0.55
NONLIVINGAREA_MEDI             0.55
NONLIVINGAREA_AVG              0.55
ELEVATORS_MODE                 0.53
ELEVATORS_MEDI              

In [36]:
# Nulos en las numéricas de train: 62 de 106 columnas tienen alguno. Hay varios grupos:
# - Vivienda (43 columnas, 48-70%): vienen en tríos _AVG/_MODE/_MEDI con el mismo % de nulos (faltan siempre juntas)
#   y son redundantes entre sí.
# - OWN_CAR_AGE (66%): hipótesis: el nulo significa "sin auto", no un dato perdido.
# - EXT_SOURCE_1 (56%) y EXT_SOURCE_3 (20%): puntajes externos de scoring, muy predictivos; hipótesis: les faltan
#   más a los thin-file.
# - AMT_REQ_CREDIT_BUREAU_* (14%, las seis iguales): parecido al % de thin-file (14,3%), a verificar.
# - DAYS_EMPLOYED (18%): son los NaN que generamos nosotros (sin_empleo).
# - Casi sin nulos (~0%): OBS/DEF_*_SOCIAL_CIRCLE, EXT_SOURCE_2, AMT_GOODS_PRICE, AMT_ANNUITY, CNT_FAM_MEMBERS,
#   DAYS_LAST_PHONE_CHANGE: acá completar con la mediana es seguro.

# pd.crosstab toma dos columnas y cuenta cuántas veces aparece cada combinación.
# ¿El nulo de OWN_CAR_AGE coincide con no tener auto (FLAG_OWN_CAR = N)? Si sí, el nulo es "no aplica", no un dato perdido.

pd.crosstab(train["FLAG_OWN_CAR"], train["OWN_CAR_AGE"].isna())

# OWN_CAR_AGE: 121.689 de sus nulos son clientes sin auto (FLAG_OWN_CAR = N): el nulo significa "no aplica".
# Solo 4 clientes con auto tienen la edad sin informar. Se completa con la mediana (de train) y FLAG_OWN_CAR
# distingue si el valor es real o de relleno. No se rellena con 0: 0 también es "auto nuevo" y se mezclarían dos
# situaciones distintas.

OWN_CAR_AGE,False,True
FLAG_OWN_CAR,,
N,0,121689
Y,62813,4


In [37]:
# % de nulos en EXT_SOURCE y en las consultas al bureau, separado por thin_file.
# Hipótesis: les faltan más a los thin-file (sin historial no hay puntaje externo).
# En criollo, quiero ver si la proporción de nulos coincide con ser thin-file.

train[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AMT_REQ_CREDIT_BUREAU_YEAR"]].isna().groupby(train["thin_file"]).mean()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,AMT_REQ_CREDIT_BUREAU_YEAR
thin_file,,,,
0,0.56,0.00,0.07,0.00
1,0.60,0.00,0.97,0.94


In [38]:
# Nulos de variables externas según thin_file:
# EXT_SOURCE_3: 7% (con historial) contra 97% (thin-file); AMT_REQ_CREDIT_BUREAU_*: 0% contra 94%.
# EXT_SOURCE_1 falta en los dos grupos (56% / 60%); EXT_SOURCE_2 (el control) casi no falta en ninguno.
# Los thin-file no solo son minoría: les faltan sistemáticamente variables muy informativas.
# Completar con la mediana los haría parecer clientes típicos en esas variables, por eso imputamos agregando una marca
# de "faltaba" (indicador de nulo). Esa marca es en parte redundante con thin_file, pero le avisa al modelo que el
# valor imputado no es un puntaje real.
# Nota: los datos no distinguen "nunca se consultó" de "se consultó y no había nada"; el nulo marca el grupo, y que
# además prediga la mora es otra pregunta (thin-file: 10,1% contra 7,7%).

# Las de vivienda son 43 columnas con nulos, así que antes de imputar vemos si los tríos _AVG/_MODE/_MEDI son
# casi la misma información, para achicar. Probamos con el trío de COMMONAREA:

train[["COMMONAREA_AVG","COMMONAREA_MODE","COMMONAREA_MEDI"]].corr()

,COMMONAREA_AVG,COMMONAREA_MODE,COMMONAREA_MEDI
COMMONAREA_AVG,1.00,0.98,0.99
COMMONAREA_MODE,0.98,1.00,0.98
COMMONAREA_MEDI,0.99,0.98,1.00


In [39]:
# ¿Todos los tríos de vivienda (_AVG, _MODE, _MEDI) están muy correlacionados? Imprimimos la correlación mínima de cada trío.
bases = [c[:-4] for c in train.columns if c.endswith("_AVG")]   # nombre base de cada trío (le saco el "_AVG")
for b in bases:
    corr_min = train[[f"{b}_AVG", f"{b}_MODE", f"{b}_MEDI"]].corr().min().min()
    print(b, round(corr_min, 2))

APARTMENTS 0.97
BASEMENTAREA 0.97
YEARS_BEGINEXPLUATATION 0.96
YEARS_BUILD 0.99
COMMONAREA 0.98
ELEVATORS 0.98
ENTRANCES 0.98
FLOORSMAX 0.99
FLOORSMIN 0.99
LANDAREA 0.98
LIVINGAPARTMENTS 0.97
LIVINGAREA 0.97
NONLIVINGAPARTMENTS 0.97
NONLIVINGAREA 0.97


In [40]:
# Correlación mínima de 0,96 a 0,99 en los 14 tríos: son casi la misma información. Dejamos solo _AVG de cada trío y
# sacamos _MODE y _MEDI (28 columnas). TOTALAREA_MODE no tiene trío y se conserva. La decisión usa solo train y no
# mira TARGET (sin leakage).

redundantes=[f"{b}_MODE" for b in bases]+[f"{b}_MEDI" for b in bases]
num_cols=[c for c in num_cols if c not in redundantes]
print(len(num_cols), len(redundantes))

# num_cols pasa de 106 a 78, y las columnas con nulos de 62 a 34. Para qué sirve:
# - eran casi copias, no aportan información nueva;
# - la capa GP de la SNGP mide distancias entre clientes: tener 3 copias de la misma variable le daría el triple de
#   peso a la vivienda en esa distancia;
# - se evitan marcas de nulo repetidas (tendríamos 43 casi idénticas) y se achica el modelo.

78 28


In [41]:
# Bloque 8. Escalado: las variables tienen escalas muy distintas (AMT_INCOME_TOTAL ~150.000, CNT_CHILDREN ~0,4,
# EXT_SOURCE_2 entre 0 y 1). La red aprende con gradientes y la capa GP mide distancias entre clientes: una variable con
# números enormes domina el cálculo sin importar si es la más importante. Escalar las pone en una escala comparable.
# XGBoost no lo necesita (corta por umbrales, así que la escala le da igual).
# StandardScaler: (x - media) / desvío, o sea media 0 y desvío 1 por columna. La media y el desvío se calculan SOLO
# con train y se aplican igual a calib y test.
# Esta celda es una primera prueba suelta: el escalado que de verdad usamos va adentro del Pipeline de la celda que
# sigue (z_train, z_calib y z_test no se usan después).

from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

scaler.fit(train[num_cols])   # aprende media y desvío de cada columna, solo con train
z_train=scaler.transform(train[num_cols])   # transforma train

# y con esa misma estructura transformamos calib y test, para evitar data leakage
z_calib=scaler.transform(calib[num_cols])
z_test=scaler.transform(test[num_cols])

In [42]:
# Bloque 9. El preprocesador completo, en un solo objeto:
# - Numéricas: mediana + marca de "faltaba" + escalado + recorte a ±5 desvíos.
# - Texto: los nulos pasan a ser la categoría "missing" + one-hot; las categorías con menos de 1.000 clientes en train
#   se juntan en una sola ("infrequent"). Con ese mínimo, la tasa de mora de cada categoría se estima con un error de
#   ~±1 punto. drop="if_binary": las variables de 2 categorías quedan en una sola columna 0/1.
# Todo se aprende con fit en train y se aplica con transform a calib y test, así nada de calib/test se filtra.
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer

num_pipe=Pipeline([ # los nombres de cada paso los ponemos para poder referirnos a ellos después
("imputar", SimpleImputer(strategy="median",add_indicator=True) ),   # imputa con la mediana de train y agrega una columna 0/1 que avisa que el valor era imputado
("escalar", StandardScaler()),
("recortar", FunctionTransformer(np.clip, kw_args={"min": -5, "max": 5}, feature_names_out="one-to-one")),
# Recorte de los valores escalados a ±5: AMT_INCOME_TOTAL (llegaba a 401 desvíos), AMT_REQ_CREDIT_BUREAU_QRT (312) y
# las *_SOCIAL_CIRCLE (75-141) tienen valores extremos reales. Recortarlos evita que dominen las distancias del GP.
# Va después del escalado (con media y desvío de train), así que tampoco hay leakage.
])

cat_pipe=Pipeline([
    ("imputar", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=1000, drop="if_binary")),   # categorías con <1000 clientes (o que no estaban en train) van a "infrequent"
])   # y si la variable es binaria no hace one-hot: deja una sola columna 0/1

# ahora juntamos las dos recetas: a cada grupo de columnas le aplicamos la suya
preproc=ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),   # a cada una le aplico su pipe
])

# fit_transform en train: preproc aprende (medianas, medias, desvíos, categorías) SOLO de train y lo transforma.
# transform en calib y test: reusa lo aprendido, sin recalcular. Así las tres matrices quedan con las mismas columnas.
X_train=preproc.fit_transform(train)
X_calib=preproc.transform(calib)
X_test=preproc.transform(test)

print(X_train.shape, X_calib.shape, X_test.shape)

# 216 columnas por ahora (más abajo sacamos 16 binarias casi constantes y quedan 200).

(184506, 216) (30751, 216) (92254, 216)


In [43]:
# Chequeo: después del preprocesamiento no debería quedar ningún nulo
import scipy.sparse as sp
X = X_train.toarray() if sp.issparse(X_train) else X_train   # por si el one-hot devolvió una matriz "rala"
print(np.isnan(X).sum())

# 0 NaN en X_train: el imputador cubrió todo (mediana en numéricas, "missing" en texto).

0


In [44]:
# Las y: el TARGET de cada parte, en el mismo orden de filas que las X.
y_train=train["TARGET"].to_numpy()
y_calib=calib["TARGET"].to_numpy()
y_test=test["TARGET"].to_numpy()

print(y_train.shape, y_calib.shape, y_test.shape)
print(y_train.mean(), y_calib.mean(), y_test.mean())

# Tasa de mora ≈ 8,07% en las tres partes: la partición estratificada mantuvo la proporción de morosos.

(184506,) (30751,) (92254,)
0.08072908198107379 0.08074534161490683 0.08072278708782275


In [45]:
# Ahora vemos los nombres de las columnas de X (mismo orden que la matriz). X_train son solo números sin etiquetas:
# con esto sabemos qué es cada columna (variable numérica, marca de "faltaba" o dummy de texto), chequeamos que el
# preprocesador armó lo esperado, y después podemos traducir "columna 143" (por ejemplo) a un nombre de variable.

nombres=preproc.get_feature_names_out()
print(len(nombres))

216


In [46]:
print(nombres[:5])
print(nombres[-5:])

# [:5] = primeros 5 elementos; [-5:] = últimos 5 (el signo menos cuenta desde el final).
# Sirve para espiar el principio y el final de una lista larga sin imprimirla entera.

['num__CNT_CHILDREN' 'num__AMT_INCOME_TOTAL' 'num__AMT_CREDIT'
 'num__AMT_ANNUITY' 'num__AMT_GOODS_PRICE']
['cat__WALLSMATERIAL_MODE_missing'
 'cat__WALLSMATERIAL_MODE_infrequent_sklearn'
 'cat__EMERGENCYSTATE_MODE_No' 'cat__EMERGENCYSTATE_MODE_Yes'
 'cat__EMERGENCYSTATE_MODE_missing']


In [47]:
# 216 columnas = numéricas (num__) + marcas de "faltaba" (missingindicator) + dummies de texto (cat__).
# Las categorías con <1000 clientes quedaron agrupadas en "infrequent_sklearn"; los nulos de texto, en "missing".

n_num = sum(n.startswith("num__") and "missingindicator" not in n for n in nombres)
n_ind = sum("missingindicator" in n for n in nombres)
n_cat = sum(n.startswith("cat__") for n in nombres)
print(n_num, n_ind, n_cat, n_num + n_ind + n_cat)

78 34 104 216


In [48]:
# 216 = 78 numéricas + 34 marcas de "faltaba" (una por cada variable que tenía nulos) + 104 dummies de texto.

# Máximo valor absoluto de cada numérica ya escalada: si algo quedó gigante, va a dominar las distancias del GP.
maximos = pd.Series(np.abs(X[:, :78]).max(axis=0), index=nombres[:78]).sort_values(ascending=False)
print(maximos.head(10))

num__CNT_CHILDREN        5.00
num__AMT_INCOME_TOTAL    5.00
num__AMT_CREDIT          5.00
num__AMT_ANNUITY         5.00
num__AMT_GOODS_PRICE     5.00
num__DAYS_EMPLOYED       5.00
num__FLAG_MOBIL          5.00
num__DAYS_REGISTRATION   5.00
num__OWN_CAR_AGE         5.00
num__LIVINGAREA_AVG      5.00
dtype: float64


In [49]:
# Tras el recorte a ±5 los máximos quedan en 5.00: los valores extremos ya no dominan. Pero antes del recorte, varias
# FLAG_* llegaban a 120-430 desvíos, y no eran outliers sino binarias casi constantes (de 1 a 13 clientes distintos
# en 184 mil). Las miramos:

flags = ["FLAG_MOBIL", "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_10", "FLAG_DOCUMENT_2", "FLAG_DOCUMENT_4"]
for c in flags:
    print(c, train[c].value_counts().to_dict())

# FLAG_MOBIL (1 cliente con 0), FLAG_DOCUMENT_12 (1), _10 (5), _2 (7), _4 (13): binarias casi constantes en train.
# Con tan pocos casos no se puede aprender nada confiable, y escaladas generan valores enormes que distorsionan las
# distancias del GP. Son candidatas a eliminar; vemos el resto de las binarias para elegir un corte.

FLAG_MOBIL {1: 184505, 0: 1}
FLAG_DOCUMENT_12 {0: 184505, 1: 1}
FLAG_DOCUMENT_10 {0: 184501, 1: 5}
FLAG_DOCUMENT_2 {0: 184499, 1: 7}
FLAG_DOCUMENT_4 {0: 184493, 1: 13}


In [50]:
binarias=[c for c in num_cols if train[c].nunique()==2 ]
conteo={}   # diccionario: así pedimos cada valor por nombre de columna y no por posición
for c in binarias:
    conteo[c]=train[c].value_counts().min()   # cuántos clientes tiene la categoría minoritaria de cada binaria

conteo=pd.Series(conteo).sort_values()   # de menor a mayor
print(len(binarias))
print(conteo.head(15))

# len(binarias): cuántas binarias hay en total (34). conteo.head(15): las 15 con menos casos en su categoría
# minoritaria; ahí se ve dónde poner el corte.

34
FLAG_MOBIL            1
FLAG_DOCUMENT_12      1
FLAG_DOCUMENT_10      5
FLAG_DOCUMENT_2       7
FLAG_DOCUMENT_4      13
FLAG_DOCUMENT_7      33
FLAG_DOCUMENT_17     42
FLAG_DOCUMENT_21     59
FLAG_DOCUMENT_20     80
FLAG_DOCUMENT_19    114
FLAG_DOCUMENT_15    214
FLAG_CONT_MOBILE    326
FLAG_DOCUMENT_14    543
FLAG_DOCUMENT_13    663
FLAG_DOCUMENT_9     695
dtype: int64


In [51]:
# Sacamos las binarias con menos de 1.000 casos en la categoría minoritaria (en train). Con tan pocos clientes no se
# puede estimar su relación con la mora, y escaladas distorsionan las distancias. Es la misma regla que el
# min_frequency=1000 de las variables de texto, y se decide solo con train.

raras=conteo[conteo<1000].index.tolist()
print(len(raras), raras)

num_cols=[c for c in num_cols if c not in raras]   # nos quedamos con las que tienen al menos 1000 casos
print(len(num_cols))

# Se eliminan 16 binarias (FLAG_MOBIL, FLAG_CONT_MOBILE y 14 FLAG_DOCUMENT_*). num_cols pasa de 78 a 62, así que hay
# que volver a ajustar el preprocesador con la lista nueva.

16 ['FLAG_MOBIL', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_21', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_15', 'FLAG_CONT_MOBILE', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_11']
62


In [52]:
# Bloque 10. Reajustamos el preprocesador con la lista nueva de num_cols (62 en vez de 78). Es el mismo
# ColumnTransformer de antes, sin las 16 binarias raras. Hay que hacerlo antes de guardar, si no se guardarían las
# matrices viejas de 216 columnas.
preproc=ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
X_train=preproc.fit_transform(train)   # aprende solo de train
X_calib=preproc.transform(calib)   # y aplica lo mismo a calib y test (sin leakage)
X_test=preproc.transform(test)

print(X_train.shape, X_calib.shape, X_test.shape)

# Quedan 200 columnas = 62 numéricas + 34 marcas de "faltaba" + 104 dummies de texto. Estas son las "200 variables"
# que ven XGBoost, la SNGP y el ensamble en todo el trabajo.

(184506, 200) (30751, 200) (92254, 200)


In [53]:
# Bloque 11. Guardamos todo en outputs/. Las matrices van como float32 (es lo que usa PyTorch y ocupa la mitad).
import joblib
import scipy.sparse as sp

def a_denso(M):
    """Si la matriz es rala (sparse), la pasa a una matriz común; si ya es común, la devuelve igual."""
    return M.toarray() if sp.issparse(M) else M

np.savez_compressed("outputs/datos_procesados.npz",
                    X_train=a_denso(X_train).astype("float32"),
                    X_calib=a_denso(X_calib).astype("float32"),
                    X_test=a_denso(X_test).astype("float32"),
                    y_train=y_train, y_calib=y_calib, y_test=y_test)
joblib.dump(preproc, "outputs/preproc.joblib")   # el preprocesador ajustado (lo usamos para leer nombres de variables)
pd.Series(preproc.get_feature_names_out()).to_csv("outputs/nombres_columnas.csv", index=False)

# Listo: X e y de train, calib y test + el preprocesador ajustado + los nombres de las columnas, todo en outputs/.
# Desde acá, todos los modelos parten de exactamente los mismos datos.